## nb_backfill_buffer

**One-time backfill** — run this once after deploying the GetFlightData Azure Function and
before enabling the `pl_hourly_predict` pipeline trigger.

Calls `POST /api/v1/flights/ingest` once per hour, going back `BACKFILL_HOURS` hours.
This uses the exact same code path as the live pipeline, so the rolling buffer is
guaranteed to be consistent with what the inference notebook will see.

The inference notebook needs 6 hours of history for the sequence input, and the lag features
(`ARR_DELAY_LAGS = [1, 3, 6, 24]`) need up to 24 hours. Without backfilling, the first
~24 hours of predictions will have degraded accuracy.

**Typical usage:** set `BACKFILL_HOURS = 168` (7 days) and run once.
Runtime: ~168 HTTP calls × ~3–5 s each ≈ 10–15 minutes.

In [ ]:
BACKFILL_HOURS        = 168   # How many hours back to fill (168 = 7 days)
FUNCTION_APP_HOSTNAME = "<replace-with-function-app-hostname>"   # e.g. flight-data-api-abc123.azurewebsites.net
FUNCTION_KEY          = "<replace-with-function-key>"            # function-level key from Azure Portal

In [ ]:
import time
import requests
import pandas as pd

INGEST_URL = f"https://{FUNCTION_APP_HOSTNAME}/api/v1/flights/ingest"
HEADERS    = {"x-functions-key": FUNCTION_KEY, "Content-Type": "application/json"}

now_utc = pd.Timestamp.utcnow().floor("h")
print(f"Backfilling {BACKFILL_HOURS} hours ending at {now_utc}")
print(f"Target URL : {INGEST_URL}")
print()

In [ ]:
errors = []

for offset in range(BACKFILL_HOURS, 0, -1):
    target_ts = now_utc - pd.Timedelta(hours=offset)
    timestamp = target_ts.strftime("%Y-%m-%dT%H:00:00Z")

    try:
        resp = requests.post(
            INGEST_URL,
            headers=HEADERS,
            json={"timestamp": timestamp},
            timeout=60,
        )
        resp.raise_for_status()
        body = resp.json()

        if body.get("status") != "ok":
            raise ValueError(f"Non-ok status: {body}")

        if offset % 24 == 0 or offset <= 5:
            rows = body.get("rows_ingested", "?")
            done = BACKFILL_HOURS - offset + 1
            print(f"  [{done}/{BACKFILL_HOURS}] {target_ts}  →  {rows:,} rows")

    except Exception as exc:
        errors.append((target_ts, str(exc)))
        print(f"  ERROR at {target_ts}: {exc}")

    # Brief pause to avoid hammering the Function App cold-start queue
    time.sleep(0.2)

print()
print(f"Backfill complete. {BACKFILL_HOURS - len(errors)}/{BACKFILL_HOURS} hours written successfully.")
if errors:
    print(f"Failed hours ({len(errors)}):")
    for ts, msg in errors:
        print(f"  {ts}  →  {msg}")
    print("Re-run with BACKFILL_HOURS set to cover only the failed range.")
else:
    print("Rolling buffer is fully populated. You can now enable pl_hourly_predict.")